# ee_verify

이미지 데이터셋 `.h5`(`h5_add_images` 출력)의 기록된 **관절 궤적**을 7d 상대 엔드이펙터(**EE-델타**)로 변환(`scripts/ee_convert.py`)한 뒤, **랜덤 N개 에피소드**를 골라 매 스텝 **mplib IK** 로 풀어 sim에서 재생하며 **성공이 그대로 재현되는지** 확인한다. GR00T용 EE 분기(Path B)의 **방법 게이트** — LeRobot로 포장(`h5_to_lerobot`)하기 전에 "EE+IK 방식이 이 task에서 통하는지" 먼저 확정한다.

`mplib`이 Linux 전용이라 실제 실행은 **WSL**에서 돌고, 이 스킬이 Windows에서 구동한다 (`task_to_h5`와 동일 패턴). `pinocchio`는 **불필요** — mplib IK로 EE→관절을 푼다.

**`run`** — 랜덤 게이트 실행 (결과 dict 반환)

| 파라미터 | 설명 | 기본값 |
|---|---|---|
| `traj_path` | 데이터셋 .h5 경로 (`obs/extra/tcp_pose` 필요) | (필수) |
| `count` | 검증할 랜덤 에피소드 수 (`0` = 전체) | `10` |
| `seed` | 랜덤 샘플 시드 (재현 가능) | `0` |
| `task` | 태스크 id. 생략 시 사이드카 `env_id` | `None` |
| `sim_backend` | 시뮬 백엔드 | `"cpu"` |

반환: `{orig_success, ee_success, reproduction_rate, ik_fails}` (문자열 값). `reproduction_rate` 가 1.0 에 가까우면 EE 방식이 건강 → `h5_to_lerobot` 진행.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from scripts.ee_verify import run

# 입력은 h5_add_images 출력(이미지 데이터셋). 원본 motionplanning.h5(obs 없음)는 안 됨.
ds = ROOT / 'data' / 'datasets' / 'ThreeColoredCubes-v1' / 'motionplanning.rgb.pd_joint_pos.physx_cpu.h5'

# 랜덤 10개를 EE-델타로 변환해 WSL mplib IK 로 재생, 성공 재현율 확인 (WSL 필요).
result = run(ds, count=10, seed=42)
result

In [ ]:
# 전체 검증이 필요하면 count=0 (느림). 게이트 통과 = 방식 신뢰 → h5_to_lerobot 은 전체를 변환한다.
# result = run(ds, count=0)

rate = float(result['reproduction_rate'])
print('reproduction_rate =', rate, '->', 'OK, 포장 진행' if rate >= 0.98 else '낮음 — EE 변환/IK 점검 필요')

CLI: `python scripts/ee_verify.py --traj-path <h5> --count 10 [--seed 42]` (`--count 0` = 전체)

동작 메모:
- **랜덤 샘플 게이트**: 변환은 결정론적이라 랜덤 N개가 재현되면 방식이 건강한 것으로 본다. 실패분 제외 필터링은 하지 않는다.
- **소스 h5 에서만 동작**: sim 을 에피소드 초기 씬으로 reset 하려면 `episode_seed` 가 필요한데, 그건 소스 데이터셋 h5 에만 있고 LeRobot 출력엔 없다. 그래서 ee_verify 는 **포장 전** 단계.
- 각 델타는 **기록된 tcp_pose 기준**으로 적용(닫힌 루프 정책이 보정하는 open-loop 드리프트는 의도적으로 배제), IK 목표는 로봇 **base 프레임**으로 변환되어 들어간다.
- 전제조건: **WSL 환경**(mplib + 소프트웨어 Vulkan). SETUP.md 참고.

보통 흐름: `h5_add_images` → `h5_report`(품질) → **`ee_verify`(EE 게이트)** → `h5_to_lerobot`(포장) → GR00T 학습(클라우드).